Notebook is used to work with Deployment scripts using PowerShell and Azure CLI when needed.  If you would like to use Cloud Shell in Notebook see ref:  [Title](https://github.com/dotnet/interactive/blob/main/samples/notebooks/powershell/Docs/Interact-With-Azure-Cloud-Shell.ipynb)

### Reference for Polyglot Notebooks: https://github.com/dotnet/interactive/blob/main/docs/magic-commands.md



In [ ]:
Install-Module -Name Az -AllowClobber -Scope CurrentUser -Force

In [ ]:
Get-AzConfig 

In [ ]:
#Login to Azure
#Use -Environment Parm to if you need to use Azure Gov.  AzureUSGovernmnet

Connect-AzAccount -Environment AzureCloud -UseDeviceAuthentication

## Deploy Managment Log Analytics Workspace

####Steps:
1. Create a Resource Group
1. Create a Log Analytics Workspace

In [ ]:
az

In [ ]:
az account list subscription

In [ ]:
# For Azure Global regions
# Set Platform management subscripion ID as the the current subscription
# Azure CLI does not run in Polyglot Notebook, run the could in terminal

# $ManagementSubscriptionId="a0d6cfbe-8dc6-49b2-80da-c2473a463a98"
# az account set --subscription $ManagementSubscriptionId

# Set the top level MG Prefix in accordance to your environment. This example assumes default 'alz'.
$TopLevelMGPrefix="alz"

# $dateYMD=$(date +%Y%m%dT%H%M%S%NZ)
$dateYMD = Get-Date -Format "yyyyMMddTHHmmssffZ"

$GROUP="rg-${TopLevelMGPrefix}-logging-001"
$NAME="alz-loggingDeployment-${dateYMD}"
$TEMPLATEFILE="S:\Repos\GitHub\csa-inabox\deploy\bicep\LandingZone\01_Logging\logging\logging.bicep"
$PARAMETERS="@S:\Repos\GitHub\csa-inabox\deploy\bicep\LandingZone\01_Logging\logging\parameters\logging.parameters.all.json"

# Create Resource Group - optional when using an existing resource group
az group create --name $GROUP --location eastus

# Deploy Module
az deployment group create --name $NAME --resource-group $GROUP --template-file $TEMPLATEFILE --parameters $PARAMETERS

In [ ]:
#Create Resource Group if not setup for data maangment zone:

$ResourceGroup = 'dmz-cloudscale-dev'

New-AzResourceGroup -Name $ResourceGroup -Location 'East US' -debug

In [ ]:
#Check Resource Group 

$ResourceGroup = 'dmz-cloudscale-dev'

Get-AzResource -ResourceGroupName $ResourceGroup | Format-Table -AutoSize

In [ ]:
#Set Param values for ARM deployment template

$params = @{
    ResourceGroupName = 'dmz-cloudscale-dev'
    TemplateFile = '..\arm\DMZ\Purview\Purview.template.json'
    TemplateParameterFile = '..\arm\DMZ\Purview\Purview.parameters.json'
    Verbose = $true
}

In [ ]:
#Test ARM deplkoyment
Test-AzResourceGroupDeployment @params


In [ ]:
#Deploy ARM Template

New-AzResourceGroupDeployment @params

In [ ]:
#Get list of Supported API versions for resource ProviderNamespace

Get-AzResourceProvider -ProviderNamespace Microsoft.Purview | Select-Object ProviderNamespace -ExpandProperty ResourceTypes | Select-Object * -ExpandProperty ApiVersions | Sort-Object -Descending

In [ ]:
#Deploy Private DNS Zones

$params = @{
    ResourceGroupName = 'demo-core-vnet'
    TemplateFile = '..\bicep\DMZ\services\Network\privateDnsZones\privateDnsZones.bicep'
}

#Test-AzResourceGroupDeployment @params

New-AzResourceGroupDeployment @params -Verbose



In [ ]:
#Test and Deploy with Bicep

$params = @{
    ResourceGroupName = 'dmz-cloudscale-dev'
    TemplateFile = '..\bicep\DMZ\services\Purview\Purview.bicep'
    Verbose = $true
}
#Test-AzResourceGroupDeployment @params

New-AzResourceGroupDeployment @params

In [ ]:
$zones = [string[]]( 'privatelink.purviewstudio.azure.com','privatelink.dfs.core.windows.net')
$rg = 'demo-core-vnet'


function GetExistingVNetLinks {
param([string[]] $zones, [string] $ResourceGroupName)
 foreach ($zone in $zones) {
            $existingLinks += Get-AzPrivateDnsVirtualNetworkLink -ZoneName $zone -ResourceGroupName $ResourceGroupName -ErrorAction 'Ignore' | Select ZoneName, @{label='vNetName'; expression={$_.VirtualNetworkId.Split('/')[-1]}} -erroraction silentlycontinue 
    }
   $existingLinks
}

GetExistingVNetLinks -zones $zones -ResourceGroupName $rg -ErrorAction 'Ignore'



In [ ]:
az storage fs directory upload -f 'inbox' -s 'C:\upload\' -d 'dotdrop/' --account-name 'mdwdatalakexet6aqs3lo2bo.blob.core.windows.net' --recursive --sas-token '?sv=2020-02-10&st=2023-09-12T15%3A24%3A12Z&se=2023-09-25T15%3A24%3A00Z&sr=d&sp=racwle&sig=ADbcXvCnnMGNTjCgz12uPGfrZGpfMnoZcxo%2BYiYsZxs%3D&sdd=1'